<a href="https://colab.research.google.com/github/IKKNIGHT/MYC-PROTAC/blob/main/OpenMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 1: Mount Drive and load config ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, time
import numpy as np

DRIVE_DIR        = '/content/drive/MyDrive/myc_gnn'
OPENMM_DRIVE_OUT = f'{DRIVE_DIR}/openmm_outputs'
os.makedirs(OPENMM_DRIVE_OUT, exist_ok=True)

with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

print(f'Candidates for OpenMM: {cfg["openmm_candidates"]}')
print(f'Top candidate:         {cfg["openmm_top_candidate"]}')
print(f'Input directory:       {cfg["openmm_input_dir"]}')

In [ ]:
# ── Cell 2: Copy inputs to local SSD ──────────────────────────────────────────
LOCAL_DIR  = '/content/myc_local'
OPENMM_IN  = '/content/openmm_inputs'
OPENMM_OUT = '/content/openmm_outputs'

for d in [LOCAL_DIR, OPENMM_IN, OPENMM_OUT]:
    os.makedirs(d, exist_ok=True)

OPENMM_DRIVE_IN = cfg['openmm_input_dir']

print('Copying inputs from Drive to local SSD...')
start  = time.time()
copied = 0
for fname in os.listdir(OPENMM_DRIVE_IN):
    if fname.endswith(('.pdb', '.fasta', '.json')):
        shutil.copy2(f'{OPENMM_DRIVE_IN}/{fname}', f'{OPENMM_IN}/{fname}')
        copied += 1
print(f'  {copied} files copied in {time.time()-start:.1f}s')

with open(f'{OPENMM_IN}/sequence_metadata.json') as f:
    seq_metadata = json.load(f)

pdb_files = sorted([f for f in os.listdir(OPENMM_IN) if f.endswith('.pdb')])
print(f'  PDB files: {len(pdb_files)}')
print(f'  Sequences: {len(seq_metadata)}')

Copying inputs from Drive to local SSD...
  81 files copied in 52.0s
  PDB files: 40
  Sequences: 40


In [ ]:
# ── Cell 3: Complete OpenMM install with CUDA support ─────────────────────────
import subprocess, sys, os

os.environ['PATH'] = (
    '/opt/miniconda/bin:/opt/miniconda/condabin:' +
    os.environ.get('PATH', '')
)
env = os.environ.copy()

def run(cmd, label=''):
    print(f'  {label or cmd[:55]}...', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True,
                       text=True, env=env)
    print('OK' if r.returncode == 0
          else f'FAILED\n{(r.stderr or r.stdout)[-300:]}')
    return r.returncode

print('GPU:')
subprocess.run('nvidia-smi | head -3', shell=True, env=env)

# ── Step 1: Miniconda ─────────────────────────────────────────────────────────
print('\nStep 1: Miniconda...')
if not os.path.exists('/opt/miniconda/bin/conda'):
    run('wget -q https://repo.anaconda.com/miniconda/'
        'Miniconda3-latest-Linux-x86_64.sh -O /tmp/mc.sh', 'Downloading')
    run('bash /tmp/mc.sh -b -p /opt/miniconda', 'Installing')
    os.environ['PATH'] = (
        '/opt/miniconda/bin:/opt/miniconda/condabin:' +
        os.environ.get('PATH', '')
    )
    env = os.environ.copy()
else:
    print('  Already installed.')

r = subprocess.run('/opt/miniconda/bin/conda --version',
                   shell=True, capture_output=True, text=True, env=env)
print(f'  {r.stdout.strip()}')

# ── Step 2: TOS ───────────────────────────────────────────────────────────────
print('\nStep 2: TOS...')
run('/opt/miniconda/bin/conda tos accept --override-channels '
    '--channel https://repo.anaconda.com/pkgs/main', 'TOS main')
run('/opt/miniconda/bin/conda tos accept --override-channels '
    '--channel https://repo.anaconda.com/pkgs/r', 'TOS r')

# ── Step 3: Create Python 3.11 env ───────────────────────────────────────────
print('\nStep 3: Creating openmm_env (Python 3.11)...')
subprocess.run(
    '/opt/miniconda/bin/conda env remove -n openmm_env -y 2>&1 | tail -2',
    shell=True, env=env, capture_output=False
)
r = subprocess.run(
    '/opt/miniconda/bin/conda create -y -n openmm_env python=3.11 '
    '--override-channels -c conda-forge 2>&1',
    shell=True, capture_output=True, text=True, env=env
)
if r.returncode != 0:
    print('FAILED:', r.stdout[-500:])
else:
    print('  openmm_env created.')

CONDA_PY = '/opt/miniconda/envs/openmm_env/bin/python'

# ── Step 4: Install OpenMM 8.0 ───────────────────────────────────────────────
print('\nStep 4: Installing OpenMM 8.0 + pdbfixer + setuptools...')
print('  (3-5 minutes — do not interrupt)')
r = subprocess.run(
    '/opt/miniconda/bin/conda install -y -n openmm_env -c conda-forge '
    'openmm=8.0 pdbfixer setuptools libstdcxx-ng 2>&1',
    shell=True, capture_output=True, text=True, env=env
)
if r.returncode != 0:
    print('FAILED:', r.stdout[-800:])
else:
    print('  OpenMM 8.0 installed.')

# ── Step 5: MDTraj + ParmEd ───────────────────────────────────────────────────
print('\nStep 5: MDTraj + ParmEd...')
run(f'{CONDA_PY} -m pip install -q mdtraj parmed --upgrade', 'Installing')

# ── Step 6: libstdc++ ────────────────────────────────────────────────────────
print('\nStep 6: Locating libstdc++...')
r = subprocess.run(
    'find /opt/miniconda/envs/openmm_env -name "libstdc++.so.6" '
    '2>/dev/null | head -1',
    shell=True, capture_output=True, text=True, env=env
)
LIBSTDCXX = r.stdout.strip()
if not LIBSTDCXX:
    print('  WARNING: libstdc++.so.6 not found')
else:
    print(f'  {LIBSTDCXX}')

env_site = '/opt/miniconda/envs/openmm_env/lib/python3.11/site-packages'
if os.path.exists(env_site) and env_site not in sys.path:
    sys.path.insert(0, env_site)
    print(f'  Added to sys.path: {env_site}')

# ── Step 7: Fix pdbfixer pkg_resources error ──────────────────────────────────
# Latest pdbfixer dropped pkg_resources — pin to 2.1 which works with OpenMM 8.0
# Also pin setuptools<70 to ensure pkg_resources module is present
print('\nStep 7: Pinning pdbfixer and setuptools...')
subprocess.run(
    f'{CONDA_PY} -m pip install "pdbfixer==2.1" --force-reinstall',
    shell=True, env=env, capture_output=False
)
subprocess.run(
    f'{CONDA_PY} -m pip install "setuptools<70" --force-reinstall',
    shell=True, env=env, capture_output=False
)

# ── Step 8: Verify via script file (avoids shell quoting issues) ──────────────
print('\nStep 8: Verifying...')

test_script = """
import openmm
from openmm import Platform
from pdbfixer import PDBFixer
import mdtraj
print("openmm", openmm.__version__)
print("pdbfixer OK")
print("mdtraj", mdtraj.__version__)
for i in range(Platform.getNumPlatforms()):
    print("Platform:", Platform.getPlatform(i).getName())
"""

with open('/tmp/test_openmm.py', 'w') as f:
    f.write(test_script)

r = subprocess.run(
    f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} /tmp/test_openmm.py',
    shell=True, capture_output=True, text=True, env=env
)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-300:])

# ── Summary ───────────────────────────────────────────────────────────────────
print('='*55)
print(f'CONDA_PY  = "{CONDA_PY}"')
print(f'LIBSTDCXX = "{LIBSTDCXX}"')

if 'pdbfixer OK' not in r.stdout:
    print('WARNING: pdbfixer not working — check STDERR above')
elif 'CUDA' in r.stdout:
    print('CUDA confirmed. Proceed to Cell 4.')
elif r.returncode == 0:
    print('OpenMM + pdbfixer working. No CUDA detected.')
    print('Switch to A100: Runtime -> Change runtime type -> A100 GPU')
else:
    print('Install failed — paste full output above.')
print('='*55)

GPU:

Step 1: Miniconda...
  Downloading... OK
  Installing... OK
  conda 26.1.1

Step 2: TOS...
  TOS main... OK
  TOS r... OK

Step 3: Creating openmm_env (Python 3.11)...
  openmm_env created.

Step 4: Installing OpenMM 8.0 + pdbfixer + setuptools...
  (3-5 minutes — do not interrupt)
  OpenMM 8.0 installed.

Step 5: MDTraj + ParmEd...
  Installing... OK

Step 6: Locating libstdc++...
  /opt/miniconda/envs/openmm_env/lib/libstdc++.so.6
  Added to sys.path: /opt/miniconda/envs/openmm_env/lib/python3.11/site-packages

Step 7: Pinning pdbfixer and setuptools...

Step 8: Verifying...
openmm 8.0
pdbfixer OK
mdtraj 1.11.1.post1
Platform: Reference
Platform: CPU
Platform: CUDA
Platform: OpenCL

CONDA_PY  = "/opt/miniconda/envs/openmm_env/bin/python"
LIBSTDCXX = "/opt/miniconda/envs/openmm_env/lib/libstdc++.so.6"
CUDA confirmed. Proceed to Cell 4.


In [ ]:
# ── Cell 4: Final — saves trajectory zip to Drive after each candidate ────────
import os, subprocess

CONDA_PY  = '/opt/miniconda/envs/openmm_env/bin/python'
LIBSTDCXX = '/opt/miniconda/envs/openmm_env/lib/libstdc++.so.6'

env = os.environ.copy()
env['PATH'] = '/opt/miniconda/bin:' + env.get('PATH', '')

PLATFORM_NAME  = 'CUDA'
PLATFORM_PROPS = "{'CudaPrecision': 'mixed'}"
DURATION_NS    = 100

SCRIPT_PATH = '/content/run_openmm.py'

script = r"""
import sys, os, json, time, zipfile
import numpy as np

LIBSTDCXX = sys.argv[4]
os.environ['LD_PRELOAD'] = LIBSTDCXX

import openmm
import openmm.app as app
import openmm.unit as unit
from openmm import Platform
from pdbfixer import PDBFixer
import mdtraj as md

PDB_PATH        = sys.argv[1]
OUT_DIR         = sys.argv[2]
NAME            = sys.argv[3]
PLATFORM_NAME   = sys.argv[5]
PLATFORM_PROPS  = eval(sys.argv[6])
DURATION_NS     = int(sys.argv[7])
DRIVE_OUT_DIR   = sys.argv[8]   # Drive output dir for this candidate

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
result = {'name': NAME, 'status': 'started'}

def save(r):
    with open(f'{OUT_DIR}/result.json', 'w') as f:
        json.dump(r, f, indent=2)

def zip_to_drive():
    zip_path = f'{DRIVE_OUT_DIR}/{NAME}.zip'
    files    = [f for f in os.listdir(OUT_DIR)
                if os.path.isfile(f'{OUT_DIR}/{f}')]
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in files:
            zf.write(f'{OUT_DIR}/{fname}', fname)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f'Zipped to Drive: {zip_path} ({size_mb:.1f} MB)', flush=True)

def make_system(topology, barostat=False):
    ff = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    system = ff.createSystem(
        topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0*unit.nanometers,
        constraints=app.HBonds,
        hydrogenMass=4*unit.amu)
    if barostat:
        system.addForce(openmm.MonteCarloBarostat(
            1*unit.atmospheres, 310*unit.kelvin, 25))
    return system

print(f'OpenMM {openmm.__version__} | {PLATFORM_NAME}', flush=True)

try:
    # ── Fix PDB ───────────────────────────────────────────────────────────────
    print('Fixing PDB...', flush=True)
    fixer = PDBFixer(filename=PDB_PATH)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.4)
    fixed_pdb = f'{OUT_DIR}/{NAME}_fixed.pdb'
    with open(fixed_pdb, 'w') as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

    # ── Solvate ───────────────────────────────────────────────────────────────
    print('Solvating...', flush=True)
    ff  = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    pdb = app.PDBFile(fixed_pdb)
    mod = app.Modeller(pdb.topology, pdb.positions)
    mod.addSolvent(ff, model='tip3p',
                   padding=1.0*unit.nanometers,
                   ionicStrength=0.15*unit.molar)
    topology  = mod.topology
    positions = mod.positions
    n_atoms   = topology.getNumAtoms()
    print(f'{n_atoms:,} atoms', flush=True)
    result['n_atoms'] = n_atoms

    plat = Platform.getPlatformByName(PLATFORM_NAME)

    # ── Minimization ──────────────────────────────────────────────────────────
    print('Minimizing...', flush=True)
    system = make_system(topology)
    intg = openmm.LangevinMiddleIntegrator(
        10*unit.kelvin, 1/unit.picosecond, 0.0005*unit.picoseconds)
    sim = app.Simulation(topology, system, intg, plat, PLATFORM_PROPS)
    sim.context.setPositions(positions)
    sim.minimizeEnergy(
        tolerance=0.1*unit.kilojoules_per_mole/unit.nanometer,
        maxIterations=50000)
    st      = sim.context.getState(getEnergy=True, getPositions=True)
    e_min   = st.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
    min_pos = st.getPositions()
    print(f'Minimized: {e_min:.1f} kJ/mol', flush=True)

    min_arr = np.array([[v.x, v.y, v.z]
                        for v in min_pos.value_in_unit(unit.nanometers)])
    if np.any(np.isnan(min_arr)):
        raise RuntimeError('NaN in minimized positions')
    print(f'Position range: '
          f'x=[{min_arr[:,0].min():.2f},{min_arr[:,0].max():.2f}] '
          f'y=[{min_arr[:,1].min():.2f},{min_arr[:,1].max():.2f}] '
          f'z=[{min_arr[:,2].min():.2f},{min_arr[:,2].max():.2f}] nm',
          flush=True)

    # ── NVT heating ───────────────────────────────────────────────────────────
    print('NVT heating...', flush=True)
    system_nvt = make_system(topology)
    intg_nvt = openmm.LangevinMiddleIntegrator(
        10*unit.kelvin, 1/unit.picosecond, 0.0005*unit.picoseconds)
    sim_nvt = app.Simulation(topology, system_nvt, intg_nvt,
                             plat, PLATFORM_PROPS)
    sim_nvt.context.setPositions(min_pos)
    sim_nvt.context.setVelocitiesToTemperature(10*unit.kelvin)

    e_check = sim_nvt.context.getState(
        getEnergy=True).getPotentialEnergy(
    ).value_in_unit(unit.kilojoules_per_mole)
    print(f'Pre-NVT energy: {e_check:.1f} kJ/mol', flush=True)

    for T in [10, 25, 50, 75, 100, 150, 200, 250, 310]:
        intg_nvt.setTemperature(T*unit.kelvin)
        sim_nvt.step(1000)
        e = sim_nvt.context.getState(getEnergy=True).getPotentialEnergy(
        ).value_in_unit(unit.kilojoules_per_mole)
        print(f'  {T}K  E={e:.0f} kJ/mol', flush=True)

    print('NVT 310K 10ps (0.5fs)...', flush=True)
    sim_nvt.step(20000)
    print('NVT 310K 20ps (1fs)...', flush=True)
    intg_nvt.setStepSize(0.001*unit.picoseconds)
    sim_nvt.step(20000)
    print('NVT 310K 20ps (2fs)...', flush=True)
    intg_nvt.setStepSize(0.002*unit.picoseconds)
    sim_nvt.step(10000)
    nvt_pos = sim_nvt.context.getState(getPositions=True).getPositions()
    print('NVT done.', flush=True)

    # ── NPT ───────────────────────────────────────────────────────────────────
    print('NPT (200ps)...', flush=True)
    system_npt = make_system(topology, barostat=True)
    intg_npt   = openmm.LangevinMiddleIntegrator(
        310*unit.kelvin, 1/unit.picosecond, 0.002*unit.picoseconds)
    sim_npt = app.Simulation(topology, system_npt, intg_npt,
                             plat, PLATFORM_PROPS)
    sim_npt.context.setPositions(nvt_pos)
    sim_npt.context.setVelocitiesToTemperature(310*unit.kelvin)
    sim_npt.step(100000)
    npt_pos = sim_npt.context.getState(getPositions=True).getPositions()
    print('NPT done.', flush=True)

    # ── Production MD ─────────────────────────────────────────────────────────
    print(f'Production MD ({DURATION_NS}ns)...', flush=True)

    traj_path     = f'{OUT_DIR}/{NAME}_prod.dcd'
    log_path      = f'{OUT_DIR}/{NAME}_prod.log'
    n_steps_total = int(DURATION_NS * 1000 / 0.002)
    save_interval = int(100 / 0.002)

    sim_npt.reporters.append(app.DCDReporter(traj_path, save_interval))
    sim_npt.reporters.append(app.StateDataReporter(
        log_path, save_interval,
        step=True, time=True, potentialEnergy=True, temperature=True,
        progress=True, totalSteps=n_steps_total + sim_npt.currentStep,
        speed=True, remainingTime=True))

    ref_pdb = f'{OUT_DIR}/{NAME}_ref.pdb'
    with open(ref_pdb, 'w') as f:
        app.PDBFile.writeFile(topology, npt_pos, f)

    ref_traj      = md.load(ref_pdb)
    peptide_atoms = ref_traj.top.select('protein and backbone')

    segment_ns    = 10
    segment_steps = int(segment_ns * 1000 / 0.002)
    n_segments    = max(1, DURATION_NS // segment_ns)
    rmsd_values   = []
    killed        = False

    for seg in range(n_segments):
        sim_npt.step(segment_steps)
        if os.path.exists(traj_path):
            try:
                traj         = md.load(traj_path, top=ref_pdb)
                traj_aligned = traj.superpose(ref_traj,
                                              atom_indices=peptide_atoms)
                rmsd_A       = float(md.rmsd(
                    traj_aligned, ref_traj,
                    atom_indices=peptide_atoms)[-1]) * 10
                rmsd_values.append(rmsd_A)
                status = 'KILLED' if rmsd_A > 4.0 else 'OK'
                print(f'Seg {seg+1}/{n_segments} '
                      f'({(seg+1)*segment_ns}ns) '
                      f'RMSD={rmsd_A:.2f}A {status}', flush=True)
                if rmsd_A > 4.0:
                    killed = True
                    break
            except Exception as e:
                print(f'RMSD check failed: {e}', flush=True)

    mean_rmsd = float(np.mean(rmsd_values)) if rmsd_values else 99.0
    passed    = not killed

    final_pdb = f'{OUT_DIR}/{NAME}_final.pdb'
    st = sim_npt.context.getState(getPositions=True)
    with open(final_pdb, 'w') as f:
        app.PDBFile.writeFile(topology, st.getPositions(), f)

    result['md_passed']  = passed
    result['mean_rmsd']  = mean_rmsd
    result['traj_path']  = traj_path
    result['final_pdb']  = final_pdb
    result['ref_pdb']    = ref_pdb
    result['status']     = 'md_complete' if passed else 'killed_rmsd'
    print(f'MD {"PASSED" if passed else "KILLED"} '
          f'mean_RMSD={mean_rmsd:.3f}A', flush=True)

    # ── MM-GBSA — protein only, no implicitSolvent arg ────────────────────────
    if passed and os.path.exists(traj_path):
        print('MM-GBSA...', flush=True)
        traj_obj     = md.load(traj_path, top=ref_pdb)
        protein_idx  = traj_obj.top.select('protein')
        traj_protein = traj_obj.atom_slice(protein_idx)
        frame_idx    = np.linspace(0, len(traj_protein)-1, 50, dtype=int)

        stripped_pdb = f'{OUT_DIR}/{NAME}_stripped.pdb'
        traj_protein[0].save_pdb(stripped_pdb)
        ref_s    = app.PDBFile(stripped_pdb)
        ff_impl  = app.ForceField('amber14-all.xml', 'implicit/obc2.xml')
        energies = []

        for fi in frame_idx:
            try:
                pos  = traj_protein.xyz[fi] * unit.nanometers
                mod2 = app.Modeller(ref_s.topology, pos)
                s2   = ff_impl.createSystem(
                    mod2.topology,
                    nonbondedMethod=app.NoCutoff)
                i2   = openmm.VerletIntegrator(0.001*unit.picoseconds)
                sim2 = app.Simulation(mod2.topology, s2, i2,
                         Platform.getPlatformByName(PLATFORM_NAME),
                         PLATFORM_PROPS)
                sim2.context.setPositions(pos)
                e = sim2.context.getState(
                    getEnergy=True).getPotentialEnergy(
                ).value_in_unit(unit.kilocalories_per_mole)
                energies.append(e)
            except:
                continue

        if energies:
            dg = float(np.mean(energies))
            result['delta_g_bind'] = dg
            result['status']       = 'mmgbsa_complete'
            print(f'DeltaG={dg:.1f} kcal/mol '
                  f'({len(energies)} frames)', flush=True)

except Exception as e:
    import traceback
    result['status'] = 'error'
    result['error']  = traceback.format_exc()
    print(f'ERROR: {e}', flush=True)
    print(traceback.format_exc(), flush=True)

save(result)

# Zip everything to Drive before exiting
print('Zipping outputs to Drive...', flush=True)
zip_to_drive()

print('DONE', flush=True)
"""

with open(SCRIPT_PATH, 'w') as f:
    f.write(script)

smoke = """
import numpy
print("numpy", numpy.__version__)
import openmm
from openmm import Platform
from pdbfixer import PDBFixer
import mdtraj
print("openmm", openmm.__version__)
print("pdbfixer OK")
print("mdtraj", mdtraj.__version__)
for i in range(Platform.getNumPlatforms()):
    print("Platform:", Platform.getPlatform(i).getName())
print("SMOKE OK")
"""
with open('/tmp/smoke.py', 'w') as f:
    f.write(smoke)

r = subprocess.run(
    f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} /tmp/smoke.py',
    shell=True, capture_output=True, text=True, env=env
)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-200:])

print('='*55)
print(f'PLATFORM_NAME  = "{PLATFORM_NAME}"')
print(f'PLATFORM_PROPS = "{PLATFORM_PROPS}"')
print(f'DURATION_NS    = {DURATION_NS}')
if 'SMOKE OK' in r.stdout:
    print('Script ready. Proceed to Cell 5.')
    if 'CUDA' not in r.stdout:
        print('NOTE: No CUDA — switch to A100 before Cell 5.')
else:
    print('Smoke test failed — check STDERR.')
print('='*55)

numpy 2.4.4
openmm 8.0
pdbfixer OK
mdtraj 1.11.1.post1
Platform: Reference
Platform: CPU
Platform: CUDA
Platform: OpenCL
SMOKE OK

PLATFORM_NAME  = "CUDA"
PLATFORM_PROPS = "{'CudaPrecision': 'mixed'}"
DURATION_NS    = 100
Script ready. Proceed to Cell 5.


In [ ]:
# ── Run this between Cell 4 and Cell 5 every session ─────────────────────────
import subprocess, os

env = os.environ.copy()
env['PATH'] = '/opt/miniconda/bin:' + env.get('PATH', '')
CONDA_PY  = '/opt/miniconda/envs/openmm_env/bin/python'
LIBSTDCXX = '/opt/miniconda/envs/openmm_env/lib/libstdc++.so.6'

# Force downgrade — ignore any conda pins
subprocess.run(
    f'{CONDA_PY} -m pip install "numpy==1.26.4" --force-reinstall '
    f'--no-deps',   # --no-deps prevents pip pulling numpy 2.x back in
    shell=True, env=env, capture_output=False
)

# Verify
r = subprocess.run(
    f'{CONDA_PY} -c "import numpy; print(numpy.__version__)"',
    shell=True, capture_output=True, text=True, env=env
)
print(f'NumPy: {r.stdout.strip()}')

# Full functional test including addMissingAtoms
test = """
import numpy
print("numpy", numpy.__version__)
import openmm
from openmm import Platform
from pdbfixer import PDBFixer
import mdtraj

# Test the exact call that was failing
fixer = PDBFixer(pdbid='1AHW')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
print("addMissingAtoms OK")

for i in range(Platform.getNumPlatforms()):
    print("Platform:", Platform.getPlatform(i).getName())
print("ALL OK")
"""
with open('/tmp/test_full.py', 'w') as f:
    f.write(test)

r = subprocess.run(
    f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} /tmp/test_full.py',
    shell=True, capture_output=True, text=True, env=env
)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-300:])

if 'ALL OK' in r.stdout and '1.' in r.stdout.split('\n')[0]:
    print('Ready. Run Cell 5.')
else:
    print('Still failing — paste output above.')

NumPy: 1.26.4
numpy 1.26.4
addMissingAtoms OK
Platform: Reference
Platform: CPU
Platform: CUDA
Platform: OpenCL
ALL OK

Ready. Run Cell 5.


In [ ]:
# ── Cell 5: Unzip from Drive on resume, zip to Drive after each run ───────────
import json, shutil, subprocess, os, time, zipfile

env = os.environ.copy()
env['PATH'] = '/opt/miniconda/bin:' + env.get('PATH', '')

OPENMM_IN        = '/content/openmm_inputs'
OPENMM_OUT       = '/content/openmm_outputs'
OPENMM_DRIVE_OUT = f'{DRIVE_DIR}/openmm_outputs'
os.makedirs(OPENMM_OUT, exist_ok=True)
os.makedirs(OPENMM_DRIVE_OUT, exist_ok=True)

SMALL_ATOM_LIMIT = 60000  # solvated atom threshold — small runs first

results      = {}
results_path = f'{OPENMM_DRIVE_OUT}/openmm_results.json'

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(f'Loaded {len(results)} existing results.')

pdb_files_all = sorted([f for f in os.listdir(OPENMM_IN)
                         if f.endswith('.pdb')])


def estimate_solvated_atoms(pdb_path):
    """
    Use actual solvated atom count from results if available.
    Otherwise estimate from dry PDB atom count × 30.
    Empirical: ~1400 dry atoms → ~37k solvated (26x)
               ~3800 dry atoms → ~118k solvated (31x)
    30x is a safe middle estimate for unknowns.
    """
    name = os.path.basename(pdb_path).replace('.pdb', '')
    if name in results and 'n_atoms' in results[name]:
        return results[name]['n_atoms']
    try:
        with open(pdb_path) as f:
            n_dry = sum(1 for l in f if l.startswith('ATOM'))
        return n_dry * 30
    except:
        return 999999


def sort_key(fname):
    name = fname.replace('.pdb', '')
    # Already completed — sort first so skip logic fires fast
    if name in results and results[name].get('status') not in (
            'started', 'error', None):
        return (0, 0)
    est = estimate_solvated_atoms(f'{OPENMM_IN}/{fname}')
    if est <= SMALL_ATOM_LIMIT:
        return (1, est)   # small: run second, smallest first
    else:
        return (2, est)   # large: run last, smallest first


pdb_files = sorted(pdb_files_all, key=sort_key)

print(f'\nProcessing order ({len(pdb_files)} candidates):')
for fname in pdb_files:
    name = fname.replace('.pdb', '')
    est  = estimate_solvated_atoms(f'{OPENMM_IN}/{fname}')
    done = name in results and results[name].get('status') not in (
           'started', 'error', None)
    tag  = 'DONE ' if done else ('small' if est <= SMALL_ATOM_LIMIT else 'LARGE')
    print(f'  {tag}  ~{est:>7,} atoms  {name}')
print()


def unzip_from_drive(name):
    """Unzip a candidate's outputs from Drive back to local SSD."""
    zip_path = f'{OPENMM_DRIVE_OUT}/{name}/{name}.zip'
    out_dir  = f'{OPENMM_OUT}/{name}'
    if os.path.exists(zip_path):
        os.makedirs(out_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(out_dir)
        print(f'  Restored from Drive zip: {name}')
        return True
    return False


def run_mmgbsa_standalone(name, traj_path, ref_pdb, out_dir):
    """Run MM-GBSA on any trajectory — protein only, no solvent args."""
    if not os.path.exists(traj_path) or not os.path.exists(ref_pdb):
        print(f'  MM-GBSA: files missing for {name}')
        return None

    mmgbsa_script = '/tmp/run_mmgbsa.py'
    code = """
import sys, os, json
import numpy as np
os.environ['LD_PRELOAD'] = sys.argv[1]
import openmm, openmm.app as app, openmm.unit as unit
from openmm import Platform
import mdtraj as md

TRAJ  = sys.argv[2]
REF   = sys.argv[3]
OUT   = sys.argv[4]
PLAT  = sys.argv[5]
PROPS = eval(sys.argv[6])

traj_full    = md.load(TRAJ, top=REF)
prot_idx     = traj_full.top.select('protein')
traj_protein = traj_full.atom_slice(prot_idx)
print(f'Frames: {len(traj_protein)}  Protein atoms: {len(prot_idx)}',
      flush=True)

stripped = OUT.replace('.json', '_stripped.pdb')
traj_protein[0].save_pdb(stripped)
ref_s    = app.PDBFile(stripped)
ff_impl  = app.ForceField('amber14-all.xml', 'implicit/obc2.xml')
plat     = Platform.getPlatformByName(PLAT)
frame_idx = np.linspace(0, len(traj_protein)-1, 50, dtype=int)
energies  = []

for i, fi in enumerate(frame_idx):
    try:
        pos  = traj_protein.xyz[fi] * unit.nanometers
        mod2 = app.Modeller(ref_s.topology, pos)
        s2   = ff_impl.createSystem(mod2.topology,
                   nonbondedMethod=app.NoCutoff)
        i2   = openmm.VerletIntegrator(0.001*unit.picoseconds)
        sim2 = app.Simulation(mod2.topology, s2, i2, plat, PROPS)
        sim2.context.setPositions(pos)
        e = sim2.context.getState(getEnergy=True).getPotentialEnergy(
        ).value_in_unit(unit.kilocalories_per_mole)
        energies.append(float(e))
        if i % 10 == 0:
            print(f'  Frame {i+1}/50: {e:.1f} kcal/mol', flush=True)
    except Exception as ex:
        print(f'  Frame {i+1} failed: {ex}', flush=True)

if energies:
    dg = float(np.mean(energies))
    print(f'DeltaG={dg:.1f} kcal/mol ({len(energies)} frames)',
          flush=True)
    with open(OUT, 'w') as f:
        json.dump({'delta_g': dg, 'n_frames': len(energies)}, f)
    print('MMGBSA_DONE', flush=True)
else:
    print('ERROR: no frames computed', flush=True)
"""
    with open(mmgbsa_script, 'w') as f:
        f.write(code)

    mmgbsa_out = f'{out_dir}/mmgbsa_result.json'
    cmd = (
        f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} {mmgbsa_script} '
        f'"{LIBSTDCXX}" "{traj_path}" "{ref_pdb}" "{mmgbsa_out}" '
        f'"{PLATFORM_NAME}" "{PLATFORM_PROPS}"'
    )
    process = subprocess.Popen(cmd, shell=True,
                               stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(f'  {line}', end='', flush=True)
    process.wait()

    if os.path.exists(mmgbsa_out):
        with open(mmgbsa_out) as f:
            data = json.load(f)
        return data.get('delta_g')
    return None


# ── Backfill MM-GBSA for passed candidates missing a score ────────────────────
print('Checking for MM-GBSA backfill...')
for name, r in list(results.items()):
    if r.get('md_passed') and 'delta_g_bind' not in r:
        print(f'\n  {name}: needs MM-GBSA — restoring from Drive...')
        out_dir = f'{OPENMM_OUT}/{name}'

        if not os.path.exists(f'{out_dir}/{name}_prod.dcd'):
            unzip_from_drive(name)

        traj_path = f'{out_dir}/{name}_prod.dcd'
        ref_pdb   = f'{out_dir}/{name}_ref.pdb'

        if not os.path.exists(traj_path):
            traj_path = f'{OPENMM_DRIVE_OUT}/{name}/{name}_prod.dcd'
            ref_pdb   = f'{OPENMM_DRIVE_OUT}/{name}/{name}_ref.pdb'
            out_dir   = f'{OPENMM_DRIVE_OUT}/{name}'

        dg = run_mmgbsa_standalone(name, traj_path, ref_pdb, out_dir)
        if dg is not None:
            results[name]['delta_g_bind'] = dg
            results[name]['status']       = 'mmgbsa_complete'
            with open(results_path, 'w') as f:
                json.dump(results, f, indent=2)
            print(f'  {name}: DeltaG={dg:.1f} kcal/mol updated.')

print('\nBackfill complete. Starting main loop...\n')

# ── Main simulation loop ──────────────────────────────────────────────────────
for pdb_fname in pdb_files:
    name      = pdb_fname.replace('.pdb', '')
    pdb_path  = f'{OPENMM_IN}/{pdb_fname}'
    out_dir   = f'{OPENMM_OUT}/{name}'
    drive_out = f'{OPENMM_DRIVE_OUT}/{name}'

    # Skip completed
    if name in results and results[name].get('status') not in (
            'started', 'error', None):
        print(f'  {name}: done ({results[name]["status"]}) — skipping')
        continue

    est_atoms = estimate_solvated_atoms(pdb_path)
    size_tag  = 'small' if est_atoms <= SMALL_ATOM_LIMIT else 'LARGE'

    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'Estimated solvated atoms: ~{est_atoms:,} ({size_tag})')
    print(f'{"="*60}')

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(drive_out, exist_ok=True)
    start = time.time()

    cmd = (
        f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} {SCRIPT_PATH} '
        f'"{pdb_path}" "{out_dir}" "{name}" '
        f'"{LIBSTDCXX}" "{PLATFORM_NAME}" '
        f'"{PLATFORM_PROPS}" {DURATION_NS} '
        f'"{drive_out}"'
    )

    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env=env
    )
    for line in process.stdout:
        print(f'  {line}', end='', flush=True)
    process.wait()

    elapsed = (time.time() - start) / 60
    print(f'\n  Elapsed: {elapsed:.1f}min  exit: {process.returncode}')

    result_file = f'{out_dir}/result.json'
    if os.path.exists(result_file):
        with open(result_file) as f:
            result = json.load(f)
    else:
        result = {'name': name, 'status': 'error',
                  'error': 'result.json not written'}

    # MM-GBSA backfill if MD passed but score missing
    if result.get('md_passed') and 'delta_g_bind' not in result:
        print(f'  Running MM-GBSA on saved trajectory...')
        traj_path = f'{out_dir}/{name}_prod.dcd'
        ref_pdb   = f'{out_dir}/{name}_ref.pdb'
        dg = run_mmgbsa_standalone(name, traj_path, ref_pdb, out_dir)
        if dg is not None:
            result['delta_g_bind'] = dg
            result['status']       = 'mmgbsa_complete'

    results[name] = result

    # Checkpoint to Drive
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    if os.path.exists(f'{out_dir}/result.json'):
        shutil.copy2(f'{out_dir}/result.json',
                     f'{drive_out}/result.json')

    status = result.get('status', 'unknown')
    dg     = result.get('delta_g_bind', 'N/A')
    rmsd   = result.get('mean_rmsd', 'N/A')
    print(f'  Status: {status} | DeltaG: {dg} | RMSD: {rmsd}')
    print(f'  Checkpointed to Drive.')

print('\nAll candidates processed.')

Loaded 40 existing results.

Processing order (40 candidates):
  DONE   ~ 93,198 atoms  candidate_1_myc_binder_88_seq24
  DONE   ~ 92,387 atoms  candidate_1_myc_binder_88_seq34
  DONE   ~ 37,172 atoms  candidate_25_myc_binder_38_seq20
  DONE   ~ 40,018 atoms  candidate_25_myc_binder_38_seq35
  DONE   ~ 56,429 atoms  candidate_2_myc_binder_66_seq32
  DONE   ~ 47,128 atoms  candidate_36_myc_binder_60_seq36
  DONE   ~ 46,960 atoms  candidate_36_myc_binder_60_seq38
  DONE   ~ 49,628 atoms  candidate_36_myc_binder_60_seq40
  DONE   ~121,348 atoms  candidate_45_myc_binder_82_seq15
  DONE   ~118,652 atoms  candidate_45_myc_binder_82_seq37
  DONE   ~125,542 atoms  candidate_51_myc_binder_98_seq10
  DONE   ~137,644 atoms  candidate_51_myc_binder_98_seq14
  DONE   ~130,630 atoms  candidate_51_myc_binder_98_seq17
  DONE   ~130,621 atoms  candidate_51_myc_binder_98_seq48
  DONE   ~ 64,384 atoms  candidate_53_myc_binder_45_seq31
  DONE   ~ 67,846 atoms  candidate_53_myc_binder_45_seq39
  DONE   ~ 8

In [ ]:
# ── Reconstruct trajectories from PDB snapshots for missing DCD files ─────────
import os, json, glob, zipfile, shutil, subprocess

env = os.environ.copy()
env['PATH'] = '/opt/miniconda/bin:' + env.get('PATH', '')
CONDA_PY  = '/opt/miniconda/envs/openmm_env/bin/python'
LIBSTDCXX = '/opt/miniconda/envs/openmm_env/lib/libstdc++.so.6'

OPENMM_OUT       = '/content/openmm_outputs'
OPENMM_DRIVE_OUT = f'{DRIVE_DIR}/openmm_outputs'
RESULTS_PATH     = f'{OPENMM_DRIVE_OUT}/openmm_results.json'

with open(RESULTS_PATH) as f:
    results = json.load(f)

# ── Write reconstruction script ───────────────────────────────────────────────
RECONSTRUCT_SCRIPT = '/tmp/reconstruct_traj.py'
code = """
import sys, os, json
import numpy as np

os.environ['LD_PRELOAD'] = sys.argv[1]

import mdtraj as md
from collections import Counter

CAND_DIR = sys.argv[2]
NAME     = sys.argv[3]
OUT_DCD  = sys.argv[4]

stage_order = ['fixed', 'minimized', 'nvt', 'npt', 'ref', 'final', 'stripped']
pdb_frames  = []
found       = []

for stage in stage_order:
    pdb_path = f'{CAND_DIR}/{NAME}_{stage}.pdb'
    if os.path.exists(pdb_path):
        try:
            t = md.load(pdb_path)
            pdb_frames.append(t)
            found.append(stage)
            print(f'Loaded {stage}: {t.n_atoms} atoms', flush=True)
        except Exception as e:
            print(f'Failed {stage}: {e}', flush=True)

if len(pdb_frames) < 1:
    print(f'ERROR: no PDB frames found in {CAND_DIR}', flush=True)
    sys.exit(1)

print(f'Found stages: {found}', flush=True)

# Filter to most common atom count so topology is consistent
atom_counts  = [t.n_atoms for t in pdb_frames]
most_common  = Counter(atom_counts).most_common(1)[0][0]
compatible   = [t for t in pdb_frames if t.n_atoms == most_common]
compat_names = [found[i] for i, t in enumerate(pdb_frames)
                if t.n_atoms == most_common]

print(f'Compatible frames ({most_common} atoms): {compat_names}', flush=True)

if len(compatible) < 1:
    print('ERROR: no compatible frames', flush=True)
    sys.exit(1)

# If only one frame, duplicate it so DCD has at least 2 frames
if len(compatible) == 1:
    compatible = compatible * 3
    print('Only 1 frame — duplicated to 3 for valid DCD', flush=True)

traj = md.join(compatible)
print(f'Combined: {len(traj)} frames', flush=True)

traj.save_dcd(OUT_DCD)
print(f'Saved DCD: {OUT_DCD}', flush=True)

ref_out = OUT_DCD.replace('_reconstructed.dcd', '_reconstructed_ref.pdb')
traj[0].save_pdb(ref_out)
print(f'Saved ref PDB: {ref_out}', flush=True)

# Report frame count to stdout for parent to parse
print(f'FRAMES={len(traj)}', flush=True)
print('RECONSTRUCT_DONE', flush=True)
"""

with open(RECONSTRUCT_SCRIPT, 'w') as f:
    f.write(code)

# ── Scan all results for missing DCDs ─────────────────────────────────────────
print('Scanning for missing trajectory files...\n')
reconstructed  = []
already_has    = []
cannot_rebuild = []

for name, r in results.items():

    local_dir = f'{OPENMM_OUT}/{name}'
    drive_dir = f'{OPENMM_DRIVE_OUT}/{name}'
    local_dcd = f'{local_dir}/{name}_prod.dcd'
    drive_dcd = f'{drive_dir}/{name}_prod.dcd'
    drive_zip = f'{drive_dir}/{name}.zip'

    # ── Check if DCD already exists ───────────────────────────────────────────
    if os.path.exists(local_dcd) or os.path.exists(drive_dcd):
        already_has.append(name)
        continue

    # ── Try unzipping from Drive first ────────────────────────────────────────
    if os.path.exists(drive_zip):
        print(f'{name}: unzipping from Drive...')
        os.makedirs(local_dir, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf:
            zf.extractall(local_dir)
        if os.path.exists(local_dcd):
            print(f'  DCD recovered from zip.')
            already_has.append(name)
            continue
        print(f'  Unzipped but DCD not in zip — will reconstruct from PDBs.')

    # ── Find available PDB files ──────────────────────────────────────────────
    os.makedirs(local_dir, exist_ok=True)

    # Copy any PDBs from Drive dir to local if not already there
    if os.path.exists(drive_dir):
        for fname in os.listdir(drive_dir):
            if fname.endswith('.pdb'):
                src = f'{drive_dir}/{fname}'
                dst = f'{local_dir}/{fname}'
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

    local_pdbs = glob.glob(f'{local_dir}/{name}_*.pdb')

    if len(local_pdbs) == 0:
        print(f'{name}: no PDB files found anywhere — cannot reconstruct')
        cannot_rebuild.append(name)
        continue

    print(f'\n{"="*55}')
    print(f'Reconstructing: {name}')
    print(f'PDBs available: {[os.path.basename(p) for p in sorted(local_pdbs)]}')

    out_dcd = f'{local_dir}/{name}_reconstructed.dcd'

    cmd = (
        f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} {RECONSTRUCT_SCRIPT} '
        f'"{LIBSTDCXX}" "{local_dir}" "{name}" "{out_dcd}"'
    )

    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    output_lines = []
    for line in process.stdout:
        print(f'  {line}', end='', flush=True)
        output_lines.append(line.strip())
    process.wait()

    if process.returncode != 0 or not os.path.exists(out_dcd):
        print(f'  Reconstruction FAILED for {name}')
        cannot_rebuild.append(name)
        continue

    # Parse frame count from output
    n_frames = 0
    for line in output_lines:
        if line.startswith('FRAMES='):
            n_frames = int(line.split('=')[1])

    ref_pdb = out_dcd.replace('_reconstructed.dcd', '_reconstructed_ref.pdb')
    print(f'  Success: {n_frames} frames')

    # Update results to point to reconstructed trajectory
    results[name]['traj_path']     = out_dcd
    results[name]['ref_pdb']       = ref_pdb
    results[name]['reconstructed'] = True
    results[name]['n_traj_frames'] = n_frames

    # Copy reconstructed files to Drive
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy2(out_dcd, f'{drive_dir}/{name}_reconstructed.dcd')
    if os.path.exists(ref_pdb):
        shutil.copy2(ref_pdb, f'{drive_dir}/{name}_reconstructed_ref.pdb')

    reconstructed.append(name)

# ── Save updated results ──────────────────────────────────────────────────────
with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\n{"="*55}')
print(f'Already had DCD:      {len(already_has)}')
print(f'Reconstructed:        {len(reconstructed)}')
print(f'Cannot reconstruct:   {len(cannot_rebuild)}')

if reconstructed:
    print(f'\nReconstructed candidates:')
    for name in reconstructed:
        n = results[name].get('n_traj_frames', '?')
        print(f'  {name}: {n} frames')
    print(f'\nNOTE: reconstructed trajectories have {n} frames vs ~700')
    print('for a full run. MM-GBSA scores are approximate — use for')
    print('relative ranking only. Flag these for full rerun if they')
    print('score well.')
    print('\nRun Cell 5 MM-GBSA backfill section to score these now.')

if cannot_rebuild:
    print(f'\nCannot reconstruct (no PDB files found):')
    for name in cannot_rebuild:
        print(f'  {name}')
    print('These candidates need a full rerun.')

Scanning for missing trajectory files...

candidate_2_myc_binder_66_seq32: unzipping from Drive...
  DCD recovered from zip.


KeyboardInterrupt: 

In [ ]:
# ── Cell 6: Filter and rank ────────────────────────────────────────────────────
import json, numpy as np

with open(f'{OPENMM_DRIVE_OUT}/openmm_results.json') as f:
    results = json.load(f)

total       = len(results)
md_passed   = [r for r in results.values() if r.get('md_passed', False)]
md_killed   = [r for r in results.values() if r.get('status') == 'killed_rmsd']
errored     = [r for r in results.values() if r.get('status') == 'error']
mmgbsa_done = [r for r in md_passed if 'delta_g_bind' in r]
ranked      = sorted(mmgbsa_done, key=lambda x: x['delta_g_bind'])

print(f'Total processed:      {total}')
print(f'Passed MD stability:  {len(md_passed)}')
print(f'Killed by RMSD:       {len(md_killed)}')
print(f'Errors:               {len(errored)}')
print(f'MM-GBSA computed:     {len(mmgbsa_done)}')

if ranked:
    print(f'\nTop 10 by binding energy:')
    print(f'{"Candidate":<45} {"DeltaG (kcal/mol)":>18} {"RMSD (A)":>10}')
    print('-' * 75)
    for r in ranked:
        print(f'{r["name"]:<45} '
              f'{r["delta_g_bind"]:>18.1f} '
              f'{r["mean_rmsd"]:>10.3f}')

# Update pipeline config
with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)
cfg['openmm_md_passed']    = len(md_passed)
cfg['openmm_md_killed']    = len(md_killed)
cfg['openmm_mmgbsa_done']  = len(mmgbsa_done)
cfg['openmm_top10_names']  = [r['name'] for r in ranked[:10]]
cfg['openmm_top10_deltag'] = [r['delta_g_bind'] for r in ranked[:10]]
with open(f'{DRIVE_DIR}/pipeline_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'\nTop {len(ranked)} saved to pipeline config.')
print(f'Proceed to Cell 7 for membrane PMF on top {len(ranked)}.')

Total processed:      40
Passed MD stability:  12
Killed by RMSD:       28
Errors:               0
MM-GBSA computed:     12

Top 10 by binding energy:
Candidate                                      DeltaG (kcal/mol)   RMSD (A)
---------------------------------------------------------------------------
candidate_75_myc_binder_92_seq33                         -5311.8      2.647
candidate_75_myc_binder_92_seq43                         -4925.2      1.920
candidate_78_myc_binder_78_seq6                          -4288.4      2.595
candidate_53_myc_binder_45_seq31                         -4263.3      3.346
candidate_78_myc_binder_78_seq25                         -4116.5      1.889
candidate_78_myc_binder_78_seq5                          -3732.3      1.993
candidate_25_myc_binder_38_seq35                         -3619.4      2.368
candidate_25_myc_binder_38_seq20                         -3310.9      2.260
candidate_9_myc_binder_96_seq8                           -3195.2      2.803
candidate_75_

In [ ]:
# ── Cell 7: Membrane PMF umbrella sampling (water stripped) ───────────────────
import json, os, subprocess, zipfile, shutil
import numpy as np

# ------------------------------------------------------------------------------
# 1. Environment & paths (assume these are set from previous cells)
# ------------------------------------------------------------------------------
env = os.environ.copy()
env['PATH'] = '/opt/miniconda/bin:' + env.get('PATH', '')

OPENMM_OUT       = '/content/openmm_outputs'
OPENMM_DRIVE_OUT = f'{DRIVE_DIR}/openmm_outputs'
PMF_SCRIPT       = '/content/run_pmf.py'
STRIP_SCRIPT     = '/tmp/strip_water.py'

# ------------------------------------------------------------------------------
# 2. Load MM‑GBSA rankings
# ------------------------------------------------------------------------------
with open(f'{OPENMM_DRIVE_OUT}/openmm_results.json') as f:
    results = json.load(f)

mmgbsa_done = [r for r in results.values() if 'delta_g_bind' in r]
ranked      = sorted(mmgbsa_done, key=lambda x: x['delta_g_bind'])
top10       = ranked[:10]

print(f'Top {len(top10)} candidates by MM-GBSA:')
for r in top10:
    print(f'  {r["name"]}: ΔG = {r["delta_g_bind"]:.1f} kcal/mol')
print()

# ------------------------------------------------------------------------------
# 3. Helper: restore final PDB from Drive / zip / local
# ------------------------------------------------------------------------------
def restore_candidate(name):
    """
    Ensure candidate files are on local SSD.
    Tries: local dir → unzip from Drive → Drive dir directly.
    Returns path to final_pdb or None.
    """
    local_dir = f'{OPENMM_OUT}/{name}'
    drive_dir = f'{OPENMM_DRIVE_OUT}/{name}'
    drive_zip = f'{drive_dir}/{name}.zip'
    local_final = f'{local_dir}/{name}_final.pdb'

    # Already local
    if os.path.exists(local_final):
        return local_final

    # Try unzipping from Drive
    if os.path.exists(drive_zip):
        print(f'  Unzipping {name} from Drive...')
        os.makedirs(local_dir, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf:
            zf.extractall(local_dir)
        if os.path.exists(local_final):
            print(f'  Restored: {local_final}')
            return local_final

    # Try copying individual file from Drive dir
    drive_final = f'{drive_dir}/{name}_final.pdb'
    if os.path.exists(drive_final):
        os.makedirs(local_dir, exist_ok=True)
        shutil.copy2(drive_final, local_final)
        print(f'  Copied from Drive: {local_final}')
        return local_final

    # Last resort — use Drive path directly
    if os.path.exists(drive_final):
        return drive_final

    return None

# ------------------------------------------------------------------------------
# 4. Helper: strip water from a PDB using mdtraj
# ------------------------------------------------------------------------------
strip_code = """
import sys, os
os.environ['LD_PRELOAD'] = sys.argv[1]
import mdtraj as md

IN_PDB  = sys.argv[2]
OUT_PDB = sys.argv[3]

print(f'Loading: {IN_PDB}', flush=True)
traj = md.load(IN_PDB)
print(f'Total atoms: {traj.n_atoms}', flush=True)

prot_idx = traj.top.select('protein')
traj_prot = traj.atom_slice(prot_idx)
print(f'Protein atoms: {traj_prot.n_atoms}', flush=True)

traj_prot[0].save_pdb(OUT_PDB)
print(f'Stripped PDB saved: {OUT_PDB}', flush=True)
print('STRIP_DONE', flush=True)
"""
with open(STRIP_SCRIPT, 'w') as f:
    f.write(strip_code)

def strip_water_from_pdb(final_pdb, name):
    """Strip water and ions from solvated final PDB, return protein-only path."""
    stripped_pdb = final_pdb.replace('_final.pdb', '_final_protein.pdb')

    # Skip if already stripped
    if os.path.exists(stripped_pdb):
        return stripped_pdb

    cmd = (
        f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} {STRIP_SCRIPT} '
        f'"{LIBSTDCXX}" "{final_pdb}" "{stripped_pdb}"'
    )
    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    for line in process.stdout:
        print(f'  {line}', end='', flush=True)
    process.wait()

    if os.path.exists(stripped_pdb):
        return stripped_pdb
    return final_pdb  # fallback to solvated if strip failed

# ------------------------------------------------------------------------------
# 5. Write the PMF simulation script (unchanged)
# ------------------------------------------------------------------------------
pmf_script = r"""
import sys, os, json
import numpy as np
import openmm
import openmm.app as app
import openmm.unit as unit
from openmm import Platform

# Setup
LIBSTDCXX = sys.argv[2]
os.environ['LD_PRELOAD'] = LIBSTDCXX
NAME, FINAL_PDB, OUT_DIR, PLATFORM_NAME = sys.argv[1], sys.argv[3], sys.argv[4], sys.argv[5]
PLATFORM_PROPS = eval(sys.argv[6])
os.makedirs(OUT_DIR, exist_ok=True)

# 1. System Prep with LARGER padding to avoid PBC artifacts
pdb = app.PDBFile(FINAL_PDB)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
mod = app.Modeller(pdb.topology, pdb.positions)
# Increased padding to 1.5nm to prevent protein from seeing its own reflection
mod.addSolvent(forcefield, padding=1.5*unit.nanometers, ionicStrength=0.15*unit.molar)

system = forcefield.createSystem(mod.topology, nonbondedMethod=app.PME,
                                 nonbondedCutoff=1.0*unit.nanometers, constraints=app.HBonds)

peptide_idx = [a.index for a in mod.topology.atoms() if a.residue.name not in ('HOH','WAT','NA','CL')]
peptide_masses = [system.getParticleMass(i).value_in_unit(unit.dalton) for i in peptide_idx]
total_mass = sum(peptide_masses)

def get_z_com(context):
    pos = context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(unit.nanometers)
    z_com = sum(pos[i][2] * peptide_masses[idx] for idx, i in enumerate(peptide_idx)) / total_mass
    return z_com

# 2. Umbrella Potential (Lowered K slightly for better sampling overlap)
k_force = 500.0
umbrella = openmm.CustomCentroidBondForce(1, "0.5*k*(z1-z0)^2")
umbrella.addGlobalParameter('k', k_force * unit.kilojoule_per_mole/unit.nanometer**2)
umbrella.addGlobalParameter('z0', 0.0 * unit.nanometer)
umbrella.addGroup(peptide_idx)
umbrella.addBond([0], [])
system.addForce(umbrella)

intg = openmm.LangevinMiddleIntegrator(310*unit.kelvin, 1/unit.picosecond, 0.002*unit.picoseconds)
sim = app.Simulation(mod.topology, system, intg, Platform.getPlatformByName(PLATFORM_NAME), PLATFORM_PROPS)
sim.context.setPositions(mod.positions)

# 3. Minimization
sim.minimizeEnergy()
initial_z = get_z_com(sim.context)

# 4. Sampling: Pulling 0.8 nm (Enough to sample local environment)
z_windows = np.linspace(initial_z, initial_z + 0.8, 12)
window_results = []

for z0 in z_windows:
    sim.context.setParameter('z0', z0)
    sim.minimizeEnergy(maxIterations=100) # Re-relax
    sim.step(5000) # Equilibration

    z_coms = []
    for _ in range(50): # Fewer, higher quality samples
        sim.step(400)
        z_coms.append(get_z_com(sim.context))

    mean_z = float(np.mean(z_coms))
    mean_force = k_force * (mean_z - z0)
    window_results.append({'z0': z0, 'mean_z': mean_z, 'mean_force': mean_force})
    print(f'  Z0={z0:.2f} | AvgZ={mean_z:.2f} | F={mean_force:.1f}', flush=True)

# 5. Integration
window_results.sort(key=lambda x: x['mean_z'])
pmf_kj = [0.0]
for i in range(1, len(window_results)):
    dz = window_results[i]['mean_z'] - window_results[i-1]['mean_z']
    f_avg = 0.5 * (window_results[i]['mean_force'] + window_results[i-1]['mean_force'])
    pmf_kj.append(pmf_kj[-1] + f_avg * dz)

pmf_kj = np.array(pmf_kj)
pmf_kj -= np.min(pmf_kj)
barrier_kcal = float(np.max(pmf_kj) / 4.184)

with open(f'{OUT_DIR}/pmf_result.json', 'w') as f:
    json.dump({'name': NAME, 'barrier_kcal': barrier_kcal, 'passes_pmf': barrier_kcal < 15.0}, f)

print(f'DONE. Barrier: {barrier_kcal:.2f} kcal/mol')
"""

with open(PMF_SCRIPT, 'w') as f:
    f.write(pmf_script)

# ------------------------------------------------------------------------------
# 6. Reset previous PMF results (they were computed with water)
# ------------------------------------------------------------------------------
pmf_path = f'{OPENMM_DRIVE_OUT}/pmf_results.json'
print('Resetting PMF results — all were computed on solvated systems...')
pmf_results = {}
with open(pmf_path, 'w') as f:
    json.dump(pmf_results, f, indent=2)
print('Reset complete. Rerunning PMF with protein‑only structures.\n')

# ------------------------------------------------------------------------------
# 7. Run PMF for each top10 candidate (with water stripping)
# ------------------------------------------------------------------------------
for candidate in top10:
    name    = candidate['name']
    pmf_out = f'{OPENMM_OUT}/{name}/pmf'

    print(f'\n{"="*60}')
    print(f'PMF: {name}')
    print(f'{"="*60}')

    # Restore solvated final PDB
    final_pdb_solvated = restore_candidate(name)
    if final_pdb_solvated is None:
        print(f'  ERROR: could not restore {name}')
        pmf_results[name] = {'passes_pmf': False,
                              'error': 'final PDB not found'}
        continue

    # Strip water — PMF needs protein‑only structure
    print(f'  Stripping water from final PDB...')
    final_pdb = strip_water_from_pdb(final_pdb_solvated, name)
    print(f'  Using: {final_pdb}')

    os.makedirs(pmf_out, exist_ok=True)

    cmd = (
        f'LD_PRELOAD={LIBSTDCXX} {CONDA_PY} {PMF_SCRIPT} '
        f'"{name}" "{LIBSTDCXX}" "{final_pdb}" "{pmf_out}" '
        f'"{PLATFORM_NAME}" "{PLATFORM_PROPS}"'
    )
    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    for line in process.stdout:
        print(f'  {line}', end='', flush=True)
    process.wait()

    result_file = f'{pmf_out}/pmf_result.json'
    if os.path.exists(result_file):
        with open(result_file) as f:
            pmf_results[name] = json.load(f)
        b = pmf_results[name]['barrier_kcal']
        p = pmf_results[name]['passes_pmf']
        print(f'  Result: {b:.1f} kcal/mol {"PASS" if p else "FAIL"}')
    else:
        pmf_results[name] = {'passes_pmf': False,
                              'error': 'script crashed'}

    # Checkpoint to Drive
    with open(pmf_path, 'w') as f:
        json.dump(pmf_results, f, indent=2)
    print(f'  Checkpointed.')

# ------------------------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------------------------
print(f'\n{"="*60}')
pmf_passing = [k for k, v in pmf_results.items()
               if v.get('passes_pmf', False)]
print(f'Passing PMF (<10 kcal/mol): {len(pmf_passing)}')
for name in pmf_passing:
    print(f'  {name}: {pmf_results[name]["barrier_kcal"]:.1f} kcal/mol')

pmf_failing = [k for k, v in pmf_results.items()
               if not v.get('passes_pmf') and 'barrier_kcal' in v]
print(f'\nFailing PMF: {len(pmf_failing)}')
for name in pmf_failing:
    print(f'  {name}: {pmf_results[name]["barrier_kcal"]:.1f} kcal/mol')

errored = [k for k, v in pmf_results.items() if 'error' in v]
if errored:
    print(f'\nErrors: {len(errored)}')
    for name in errored:
        print(f'  {name}: {pmf_results[name]["error"]}')

# Final save
with open(pmf_path, 'w') as f:
    json.dump(pmf_results, f, indent=2)
print(f'\nPMF results saved: {pmf_path}')

Top 10 candidates by MM-GBSA:
  candidate_75_myc_binder_92_seq33: ΔG = -5311.8 kcal/mol
  candidate_75_myc_binder_92_seq43: ΔG = -4925.2 kcal/mol
  candidate_78_myc_binder_78_seq6: ΔG = -4288.4 kcal/mol
  candidate_53_myc_binder_45_seq31: ΔG = -4263.3 kcal/mol
  candidate_78_myc_binder_78_seq25: ΔG = -4116.5 kcal/mol
  candidate_78_myc_binder_78_seq5: ΔG = -3732.3 kcal/mol
  candidate_25_myc_binder_38_seq35: ΔG = -3619.4 kcal/mol
  candidate_25_myc_binder_38_seq20: ΔG = -3310.9 kcal/mol
  candidate_9_myc_binder_96_seq8: ΔG = -3195.2 kcal/mol
  candidate_75_myc_binder_92_seq27: ΔG = -2911.1 kcal/mol

Resetting PMF results — all were computed on solvated systems...
Reset complete. Rerunning PMF with protein‑only structures.


PMF: candidate_75_myc_binder_92_seq33
  Stripping water from final PDB...
  Using: /content/openmm_outputs/candidate_75_myc_binder_92_seq33/candidate_75_myc_binder_92_seq33_final_protein.pdb
    Z0=6.54 | AvgZ=6.58 | F=21.0
    Z0=6.61 | AvgZ=6.56 | F=-22.5
    Z0=6

In [ ]:
# ── Cell 8: Final candidates and Drive push ────────────────────────────────────
import json, shutil, os

with open(f'{OPENMM_DRIVE_OUT}/openmm_results.json') as f:
    results = json.load(f)
with open(f'{OPENMM_DRIVE_OUT}/pmf_results.json') as f:
    pmf_results = json.load(f)

final = []
for name, pmf in pmf_results.items():
    if not pmf.get('passes_pmf', False):
        continue
    md_r = results.get(name, {})
    if not md_r.get('md_passed', False):
        continue
    final.append({
        'name':        name,
        'rmsd':        md_r.get('mean_rmsd', 99.0),
        'delta_g':     md_r.get('delta_g_bind', 99.0),
        'pmf_barrier': pmf.get('barrier_kcal', 99.0),
    })

final.sort(key=lambda x: x['delta_g'])

print(f'Final synthesis candidates: {len(final)}')
if final:
    print(f'\n{"Candidate":<45} {"DeltaG":>8} {"PMF":>8} {"RMSD":>8}')
    print('-' * 71)
    for c in final:
        print(f'{c["name"]:<45} '
              f'{c["delta_g"]:>8.1f} '
              f'{c["pmf_barrier"]:>8.1f} '
              f'{c["rmsd"]:>8.3f}')

# Save final PDBs to synthesis folder
SYNTH_DIR = f'{DRIVE_DIR}/synthesis_candidates'
os.makedirs(SYNTH_DIR, exist_ok=True)
for c in final:
    src = f'{OPENMM_OUT}/{c["name"]}/{c["name"]}_final.pdb'
    if os.path.exists(src):
        shutil.copy2(src, f'{SYNTH_DIR}/{c["name"]}.pdb')
        print(f'  Saved: {c["name"]}.pdb')

# Update pipeline config
with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)
cfg['synthesis_candidates']        = [c['name'] for c in final]
cfg['synthesis_candidates_deltag'] = [c['delta_g'] for c in final]
cfg['synthesis_candidates_pmf']    = [c['pmf_barrier'] for c in final]
with open(f'{DRIVE_DIR}/pipeline_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'\nPipeline config updated.')
print(f'Phase III complete — {len(final)} candidates ready for synthesis.')

Final synthesis candidates: 8

Candidate                                       DeltaG      PMF     RMSD
-----------------------------------------------------------------------
candidate_75_myc_binder_92_seq33               -5311.8     10.5    2.647
candidate_75_myc_binder_92_seq43               -4925.2      2.5    1.920
candidate_78_myc_binder_78_seq6                -4288.4      8.6    2.595
candidate_53_myc_binder_45_seq31               -4263.3      7.6    3.346
candidate_78_myc_binder_78_seq5                -3732.3     14.0    1.993
candidate_25_myc_binder_38_seq20               -3310.9      9.2    2.260
candidate_9_myc_binder_96_seq8                 -3195.2     10.3    2.803
candidate_75_myc_binder_92_seq27               -2911.1      7.1    1.899
  Saved: candidate_75_myc_binder_92_seq33.pdb
  Saved: candidate_75_myc_binder_92_seq43.pdb
  Saved: candidate_78_myc_binder_78_seq6.pdb
  Saved: candidate_53_myc_binder_45_seq31.pdb
  Saved: candidate_78_myc_binder_78_seq5.pdb
  Saved: can

In [ ]:
import json

with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

cfg['phase3_complete']        = True
cfg['synthesis_candidates']   = [
    'candidate_75_myc_binder_92_seq43',   # lead
    'candidate_75_myc_binder_92_seq27',   # lead
    'candidate_78_myc_binder_78_seq6',    # strong
    'candidate_53_myc_binder_45_seq31',   # strong
    'candidate_25_myc_binder_38_seq20',   # recovered
    'candidate_78_myc_binder_78_seq5',    # marginal PMF
    'candidate_9_myc_binder_96_seq8',     # borderline
    'candidate_75_myc_binder_92_seq33',   # borderline PMF
]
cfg['lead_candidate']         = 'candidate_75_myc_binder_92_seq43'
cfg['lead_pmf_kcal']          = 2.5
cfg['lead_rmsd_A']            = 1.92
cfg['lead_delta_g']           = -4925.2

with open(f'{DRIVE_DIR}/pipeline_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Pipeline config updated. Phase III complete.')

Pipeline config updated. Phase III complete.
